# Resultados e Comparacao dos Agentes

Este notebook esta alinhado com a versao simplificada do projeto: o estado tem 6 dimensoes (`retornos dos 3 ativos + pesos atuais`) e a recompensa combina retorno liquido, bonus de diversificacao e penalidade de concentracao.

As analises abaixo focam no que os agentes realmente usam e aprendem: reward, retorno, erro TD, exploracao, estados visitados e alocacao da carteira.

In [ ]:
from pathlib import Path
import contextlib
import io
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "main.py").exists() and (candidate / "main_td.py").exists():
            return candidate
    raise RuntimeError("Raiz do projeto nao encontrada.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import main as q_main
import main_td
from ambiente.portfolio_env import PortfolioEnv
from agentes.Q_learning import AgentQLearning
from agentes.TD_learning import AgentTD
from helpers.data_loader import load_train_data, load_test_data

RESULTS_DIR = ROOT / "resultados"
FIGURES_DIR = RESULTS_DIR / "figuras"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_TRAINING = True
MAIN_EPISODES = q_main.N_EPISODES
ALPHA_VALUES = [0.02, 0.05, 0.1, 0.2]
ALPHA_EXPERIMENT_EPISODES = 100
SMOOTH_WINDOW = 20

COLORS = {
    "q": "#2563eb",
    "td": "#dc2626",
    "buy_hold": "#16a34a",
    "cdi": "#7c3aed",
}

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})


def load_json(path):
    return json.loads(Path(path).read_text())


def load_saved_results():
    q_metrics = load_json(RESULTS_DIR / "metrics_qlearning.json")
    td_metrics = load_json(RESULTS_DIR / "metrics_td.json")
    q_history = load_json(RESULTS_DIR / "training_history.json")
    td_history = load_json(RESULTS_DIR / "training_history_td.json")
    q_eval = load_json(RESULTS_DIR / "eval_portfolio.json")
    td_eval = load_json(RESULTS_DIR / "eval_portfolio_td.json")
    return q_metrics, td_metrics, q_history, td_history, q_eval, td_eval


def smooth(values, window=20):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.array([]), np.array([])
    if len(values) < window:
        return np.arange(1, len(values) + 1), values
    kernel = np.ones(window) / window
    return np.arange(window, len(values) + 1), np.convolve(values, kernel, mode="valid")


def cumulative_return(values, initial=None):
    values = np.asarray(values, dtype=float)
    if initial is None:
        initial = values[0]
    return values / initial - 1.0


def portfolio_allocation_stats(eval_data, env_config):
    weights = np.asarray(eval_data["weights_history"], dtype=float)
    hhi = np.sum(weights ** 2, axis=1)
    max_weight = np.max(weights, axis=1)
    bonus = env_config["alpha_diversification"] * (1.0 - hhi)
    penalty = env_config["beta_concentration"] * np.maximum(
        0.0, max_weight - env_config["concentration_threshold"]
    )
    return {
        "hhi_medio": float(np.mean(hhi)),
        "diversificacao_media": float(np.mean(1.0 - hhi)),
        "peso_maximo_medio": float(np.mean(max_weight)),
        "bonus_medio": float(np.mean(bonus)),
        "penalidade_media": float(np.mean(penalty)),
    }


def greedy_portfolio_eval(agent, env):
    state = agent.discretize(env.reset())
    results = {
        "portfolio_values": [env.portfolio_value],
        "weights_history": [env.weights.copy()],
        "rewards": [],
        "returns": [],
        "actions": [],
    }
    done = False
    while not done:
        action = int(np.argmax(agent.q_table[state]))
        next_state_continuous, reward, done, info = env.step(action)
        state = agent.discretize(next_state_continuous)
        results["portfolio_values"].append(float(info["portfolio_value"]))
        results["weights_history"].append(info["weights"].tolist())
        results["rewards"].append(float(reward))
        results["returns"].append(float(info["portfolio_return"]))
        results["actions"].append(action)
    final_value = results["portfolio_values"][-1]
    results["final_value"] = final_value
    results["total_return"] = final_value / env.initial_balance - 1.0
    return results


print(f"Raiz do projeto: {ROOT}")
print(f"Resultados: {RESULTS_DIR}")

## 1. Treinamento

Esta celula executa os pipelines principais e atualiza os arquivos JSON em `resultados/`. A saida detalhada dos scripts e suprimida para manter o notebook focado nas analises.

In [ ]:
def run_main_quiet(module, label, n_episodes):
    original_episodes = module.N_EPISODES
    original_interval = module.LOG_INTERVAL
    module.N_EPISODES = n_episodes
    module.LOG_INTERVAL = max(1, n_episodes // 5)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            module.main()
    finally:
        module.N_EPISODES = original_episodes
        module.LOG_INTERVAL = original_interval
    print(f"{label}: treino concluido com {n_episodes} episodios.")


if RUN_TRAINING:
    run_main_quiet(q_main, "Q-Learning", MAIN_EPISODES)
    run_main_quiet(main_td, "TD-Learning", MAIN_EPISODES)
else:
    print("Treinamento pulado. Usando JSONs existentes em resultados/.")

## 2. Comparacao Final

A tabela compara os agentes e benchmarks usando valor final, retorno total e estatisticas de treinamento ligadas ao aprendizado tabular.

In [ ]:
q_metrics, td_metrics, q_history, td_history, q_eval, td_eval = load_saved_results()


def build_comparison_table(q_metrics, td_metrics, q_history, td_history):
    q_bench = q_metrics["benchmarks"]
    rows = [
        {
            "estrategia": "Q-Learning",
            "valor_final": q_metrics["agent"]["final_value"],
            "retorno_total": q_metrics["agent"]["total_return"],
            "reward_medio_final": q_metrics["training"]["avg_reward_last_50"],
            "td_error_final": q_history["episode_td_errors"][-1],
            "td_error_medio_final": float(np.mean(q_history["episode_td_errors"][-50:])),
            "estados_visitados": q_metrics["training"]["n_states_visited"],
            "epsilon_final": q_metrics["training"]["final_epsilon"],
            "alpha": q_metrics["training"]["final_alpha"],
        },
        {
            "estrategia": "TD-Learning",
            "valor_final": td_metrics["agent"]["final_value"],
            "retorno_total": td_metrics["agent"]["total_return"],
            "reward_medio_final": td_metrics["training"]["avg_reward_last_50"],
            "td_error_final": td_history["episode_td_errors"][-1],
            "td_error_medio_final": float(np.mean(td_history["episode_td_errors"][-50:])),
            "estados_visitados": td_metrics["training"]["n_states_visited"],
            "epsilon_final": td_metrics["training"]["final_epsilon"],
            "alpha": td_metrics["training"]["final_alpha"],
        },
        {
            "estrategia": "Buy&Hold 1/3",
            "valor_final": q_bench["buy_hold"]["final_value"],
            "retorno_total": q_bench["buy_hold"]["total_return"],
            "reward_medio_final": np.nan,
            "td_error_final": np.nan,
            "td_error_medio_final": np.nan,
            "estados_visitados": np.nan,
            "epsilon_final": np.nan,
            "alpha": np.nan,
        },
        {
            "estrategia": "CDI",
            "valor_final": q_bench["cdi"]["final_value"],
            "retorno_total": q_bench["cdi"]["total_return"],
            "reward_medio_final": np.nan,
            "td_error_final": np.nan,
            "td_error_medio_final": np.nan,
            "estados_visitados": np.nan,
            "epsilon_final": np.nan,
            "alpha": np.nan,
        },
    ]
    return pd.DataFrame(rows)


comparison_df = build_comparison_table(q_metrics, td_metrics, q_history, td_history)
display(comparison_df.style.format({
    "valor_final": "R$ {:,.2f}",
    "retorno_total": "{:.2%}",
    "reward_medio_final": "{:.4f}",
    "td_error_final": "{:.6f}",
    "td_error_medio_final": "{:.6f}",
    "estados_visitados": "{:,.0f}",
    "epsilon_final": "{:.4f}",
    "alpha": "{:.4f}",
}))

## 3. Convergencia de Treinamento

As curvas abaixo mostram se o sinal de aprendizado estabiliza: reward acumulado, valor final por episodio, erro TD e reducao de exploracao.

In [ ]:
def plot_training_convergence(q_history, td_history, q_metrics, td_metrics, window=20):
    episodes = np.arange(1, len(q_history["episode_rewards"]) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    panels = [
        (axes[0, 0], "episode_rewards", "Reward acumulado", "Reward"),
        (axes[0, 1], "episode_portfolio_values", "Valor final por episodio", "Valor (R$)"),
        (axes[1, 0], "episode_td_errors", "Erro TD medio", "|erro TD|"),
        (axes[1, 1], "episode_epsilons", "Exploracao", "epsilon"),
    ]

    for ax, key, title, ylabel in panels:
        ax.plot(episodes, q_history[key], color=COLORS["q"], alpha=0.12)
        ax.plot(episodes, td_history[key], color=COLORS["td"], alpha=0.12)
        q_x, q_y = smooth(q_history[key], window)
        td_x, td_y = smooth(td_history[key], window)
        ax.plot(q_x, q_y, color=COLORS["q"], linewidth=2, label="Q-Learning")
        ax.plot(td_x, td_y, color=COLORS["td"], linewidth=2, label="TD-Learning")
        ax.set_title(title)
        ax.set_xlabel("Episodio")
        ax.set_ylabel(ylabel)
        ax.legend()

    alpha_q = q_metrics["training"]["final_alpha"]
    alpha_td = td_metrics["training"]["final_alpha"]
    axes[1, 1].text(
        0.02,
        0.05,
        f"alpha fixo: Q={alpha_q:.3f} | TD={alpha_td:.3f}",
        transform=axes[1, 1].transAxes,
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.show()


plot_training_convergence(q_history, td_history, q_metrics, td_metrics, SMOOTH_WINDOW)

## 4. Teste e Alocacao

Agora comparamos o desempenho fora do treino e analisamos a carteira aprendida por pesos, HHI, diversificacao e concentracao media.

In [ ]:
def plot_test_comparison(q_eval, td_eval):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].plot(q_eval["portfolio_values"], label="Q-Learning", color=COLORS["q"])
    axes[0].plot(td_eval["portfolio_values"], label="TD-Learning", color=COLORS["td"])
    axes[0].plot(q_eval["benchmark_buy_hold"], label="Buy&Hold 1/3", color=COLORS["buy_hold"], linestyle="--")
    axes[0].plot(q_eval["benchmark_cdi"], label="CDI", color=COLORS["cdi"], linestyle="--")
    axes[0].set_title("Valor do portfolio no teste")
    axes[0].set_xlabel("Passo")
    axes[0].set_ylabel("Valor (R$)")
    axes[0].legend()

    series = {
        "Q-Learning": q_eval["portfolio_values"],
        "TD-Learning": td_eval["portfolio_values"],
        "Buy&Hold 1/3": q_eval["benchmark_buy_hold"],
        "CDI": q_eval["benchmark_cdi"],
    }
    for label, values in series.items():
        color_key = "q" if label == "Q-Learning" else "td" if label == "TD-Learning" else "buy_hold" if label.startswith("Buy") else "cdi"
        axes[1].plot(cumulative_return(values), label=label, color=COLORS[color_key])
    axes[1].axhline(0, color="gray", linestyle=":")
    axes[1].set_title("Retorno acumulado no teste")
    axes[1].set_xlabel("Passo")
    axes[1].set_ylabel("Retorno acumulado")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_weights(eval_data, label, color):
    weights = np.asarray(eval_data["weights_history"], dtype=float)
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.stackplot(
        np.arange(len(weights)),
        weights[:, 0],
        weights[:, 1],
        weights[:, 2],
        labels=["ITUB4", "BOVA11", "BBAS3"],
        colors=["#60a5fa", "#34d399", "#f59e0b"],
        alpha=0.85,
    )
    ax.set_ylim(0, 1)
    ax.set_title(f"Pesos da carteira no teste - {label}")
    ax.set_xlabel("Passo")
    ax.set_ylabel("Peso")
    ax.legend(loc="upper left", ncol=3)
    plt.tight_layout()
    plt.show()


plot_test_comparison(q_eval, td_eval)
plot_weights(q_eval, "Q-Learning", COLORS["q"])
plot_weights(td_eval, "TD-Learning", COLORS["td"])

allocation_df = pd.DataFrame([
    {"algoritmo": "Q-Learning", **portfolio_allocation_stats(q_eval, q_metrics["config"]["env"])},
    {"algoritmo": "TD-Learning", **portfolio_allocation_stats(td_eval, td_metrics["config"]["env"])},
])
display(allocation_df.style.format({
    "hhi_medio": "{:.4f}",
    "diversificacao_media": "{:.4f}",
    "peso_maximo_medio": "{:.2%}",
    "bonus_medio": "{:.6f}",
    "penalidade_media": "{:.6f}",
}))

## 5. Sensibilidade a `alpha`

Como `alpha` agora e fixo durante o treino, a sensibilidade abaixo compara valores constantes de taxa de aprendizado para cada algoritmo.

In [ ]:
def run_alpha_experiment(agent_cls, label, alpha_values, n_episodes=100):
    train_df = load_train_data()
    test_df = load_test_data()
    base_agent_config = q_main.AGENT_CONFIG.copy() if agent_cls is AgentQLearning else main_td.AGENT_CONFIG.copy()
    env_config = q_main.ENV_CONFIG.copy()

    records = []
    for alpha_value in alpha_values:
        config = base_agent_config.copy()
        config["alpha"] = alpha_value
        config["epsilon"] = base_agent_config["epsilon"]

        print(f"{label}: alpha={alpha_value} | episodios={n_episodes}")
        init_env = PortfolioEnv(train_df, **env_config)
        agent = agent_cls(init_env, **config)
        history = agent.train(
            PortfolioEnv(train_df, **env_config),
            n_episodes=n_episodes,
            log_interval=max(1, n_episodes),
        )
        eval_results = greedy_portfolio_eval(agent, PortfolioEnv(test_df, **env_config))

        records.append({
            "algoritmo": label,
            "alpha": alpha_value,
            "valor_final": eval_results["final_value"],
            "retorno_total": eval_results["total_return"],
            "reward_medio_final": float(np.mean(history["episode_rewards"][-50:])),
            "td_error_final": history["episode_td_errors"][-1],
            "td_error_medio_final": float(np.mean(history["episode_td_errors"][-50:])),
            "estados_visitados": history["n_states_visited"],
        })

    return pd.DataFrame(records)


def plot_alpha_experiment(alpha_df):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    metrics = [
        ("retorno_total", "Retorno total"),
        ("valor_final", "Valor final"),
        ("reward_medio_final", "Reward medio final"),
        ("td_error_medio_final", "Erro TD medio final"),
    ]
    for ax, (column, title) in zip(axes.flat, metrics):
        for algoritmo, group in alpha_df.groupby("algoritmo"):
            color = COLORS["q"] if algoritmo == "Q-Learning" else COLORS["td"]
            ax.plot(group["alpha"], group[column], marker="o", linewidth=2, label=algoritmo, color=color)
        ax.set_title(title)
        ax.set_xlabel("alpha")
        ax.legend()
    plt.tight_layout()
    plt.show()


q_alpha = run_alpha_experiment(AgentQLearning, "Q-Learning", ALPHA_VALUES, ALPHA_EXPERIMENT_EPISODES)
td_alpha = run_alpha_experiment(AgentTD, "TD-Learning", ALPHA_VALUES, ALPHA_EXPERIMENT_EPISODES)
alpha_df = pd.concat([q_alpha, td_alpha], ignore_index=True)

display(alpha_df.style.format({
    "alpha": "{:.3f}",
    "valor_final": "R$ {:,.2f}",
    "retorno_total": "{:.2%}",
    "reward_medio_final": "{:.4f}",
    "td_error_final": "{:.6f}",
    "td_error_medio_final": "{:.6f}",
    "estados_visitados": "{:,.0f}",
}))
plot_alpha_experiment(alpha_df)

## Leitura Recomendada

- Se o reward estabiliza e o erro TD cai, a Q-table esta ficando mais consistente.
- Se o retorno no teste nao acompanha o reward de treino, o agente pode estar aprendendo um padrao especifico do periodo de treino.
- HHI menor e diversificacao media maior indicam carteiras menos concentradas, que agora fazem parte direta do desenho da recompensa.